In [ ]:
class AcademicAdvisorAgent:
    def __init__(self):
        # KB - knowledge base
        self.KB = {
            "weights": {
                "attendance": 0.3,
                "coursework": 0.4,
                "deadline": 0.2,
                "difficulty": 0.1
            },

            # Prior (assujmed) probabilities (before seeing evidence)
            "priors": {
                "high_risk": 0.3,
                "medium_risk": 0.4,
                "low_risk": 0.3
            },

            # Rules (explicit reasoning knowledge)
            "rules": [
                "low_attendance -> higher_risk",
                "low_coursework -> higher_risk",
                "deadline_close -> higher_risk",
                "high_performance -> lower_risk"
            ]
        }
    # def __init__(self):
    #     # wegiths represent importance of each factor
    #     self.weights = {
    #         "attendance": 0.3,
    #         "coursework": 0.4,
    #         "deadline": 0.2,
    #         "difficulty": 0.1
    #     }

    def interpret_input(self, student_data):
        return student_data
    
    # inference (weighted)
    def infer_risk(self, data):
        # this function calculates how risky the student's situation is
        # takes student data -> calculates risk score -> decides risk level -> explains why -> gives confidence

        explanation = [] # stores reasons 
        score = 0 # total risk amount
        total_weight = 0 # how much data we used

        # attendance
        if "attendance" in data:
            att = data["attendance"]
            if att < 50:
                score += self.weights["attendance"] * 1.0
                explanation.append("Very low attendance")
            elif att < 70:
                score += self.weights["attendance"] * 0.5
                explanation.append("Moderate attendance")
            else:
                explanation.append("Good attendance")
            total_weight += self.weights["attendance"]


        # coursework
        if "coursework" in data:
            cw = data["coursework"]
            if cw < 50:
                score += self.weights["coursework"] * 1.0
                explanation.append("Low coursework score")

            elif cw < 65:
                score += self.weights["coursework"] * 0.5
                explanation.append("Average coursework")
            else:
                explanation.append("Strong coursework performance")
            total_weight += self.weights["coursework"]

        
        # deadline proximity
        if "deadline" in data:
            d = data["deadline"]
            if d <= 3:
                score += self.weights["deadline"] * 1.0
                explanation.append("Deadline very close")
            elif d <= 7:
                score += self.weights["deadline"] * 0.5
                explanation.append("Deadline approaching")
            total_weight += self.weights["deadline"]


        # difficulty perception
        if "difficulty" in data:
            if data["difficulty"] == "hard":
                score += self.weights["difficulty"] * 1.0
                explanation.append("Subject perceived as difficult")
            elif data["difficulty"] == "medium":
                score += self.weights["difficulty"] * 0.5
                explanation.append("Moderate difficulty")
            total_weight += self.weights["difficulty"]

        # normalize score (0 to 1)
        # convert score into a value between 0 and 1
        if total_weight == 0:
            return "unknown", 0.3, ["No useful data provided"]
        
        risk_score = score / total_weight

        # risk classification
        if risk_score > 0.7:
            risk = "high_risk"
        elif risk_score > 0.4:
            risk = "medium_risk"
        else:
            risk = "low_risk"

        # confidence = how much data we used 
        confidence = total_weight # more data = higher confidence
        return risk, round(confidence, 2), explanation



    # decision logic
    def decide_action(self, risk_level, confidence):
        # takes the risk level and returns the advice message

        if confidence < 0.5:
            return "I am not confident due to limited information. Please provide more details."
        

        if risk_level == "high_risk":
            return "You are at high risk. Prioritise urgent revision and seek academic support"
        elif risk_level == "medium_risk":
            return "You are at moderate risk. Improve consistency and focus on weak areas"
        elif risk_level == "low_risk":
            return "You are performing well. Maintain your study strategy"
        else:
            return "I need more information to assess your sitation"


    # explainability
    def explain(self, explanation, confidence, risk):
        return (
            f"Risk level: {risk}\n"
            f"Reasoning: {', '.join(explanation)}\n"
            f"Confidence: {confidence}"
        )
    
    # bayesian risk estimate function
    def bayesian_risk_estimate(self, data):
        # estimate probabilities for each risk level
        # how likely is each risk level given the student data
        # update belief when you see evidence
        
        # assumed prior probabilities
        p_high = 0.3
        p_medium = 0.4
        p_low = 0.3

        # heuristics:
        # 1.5 - strong signal
        # 1.4 - medium signal
        # 1.2 - weaker signal

        # update probabilities based on evidence
        # attendance effect
        if "attendance" in data:
            att = data["attendance"]
            if att < 50:
                p_high *= 1.5 # low attendance => high risk (more likely)
                p_medium *= 1.2
            elif att > 75:
                p_low *= 1.5

        # coursework effect
        if "coursework" in data:
            cw = data["coursework"]
            if cw < 50:
                p_high *= 1.5
            elif cw > 70:
                p_low *= 1.5

        # deadline effect
        if "deadline" in data:
            d = data["deadline"]
            if d <= 3:
                p_high *= 1.4
            elif d > 7:
                p_low *= 1.2

        # normalize probabilities
        # We divide each value by the total so all probabilities sum to 1
        total = p_high + p_medium + p_low

        p_high /= total
        p_medium /= total
        p_low /= total

        return {
            "high_risk": round(p_high, 2),
            "medium_risk": round(p_medium, 2),
            "low_risk": round(p_low, 2)
        }
    
    # full pipeline
    def run(self, student_data):
        data = self.interpret_input(student_data)
        risk, confidence, explanation = self.infer_risk(data)

        # bayesian probabilities
        probabilities = self.bayesian_risk_estimate(data)

        decision = self.decide_action(risk, confidence)
        explanation_text = self.explain(explanation, confidence, risk)

        return {
            "decision": decision,
            "explanation": explanation_text,
            "probabilities": probabilities
        }
    


In [12]:
agent = AcademicAdvisorAgent()

def print_result(student_name, student_data):
    print(f"Advice for {student_name}:")
    
    result = agent.run(student_data)
    
    print("Decision:")
    print(result["decision"])
    
    print("\nExplanation:")
    print(result["explanation"])
    
    print("\nProbabilities:")
    print(result["probabilities"])
    
    print("----------------------------------------------------------")


# Test cases
student_1 = {
    "attendance": 40,
    "coursework": 45,
    "deadline": 2,
    "difficulty": "hard"
}

student_2 = {"attendance": 30, "coursework": 40, "deadline": 1}
student_3 = {"attendance": 65, "coursework": 60, "deadline": 5}
student_4 = {"attendance": 85, "coursework": 80, "deadline": 10}
student_5 = {"attendance": 80}


# Run tests
print_result("student 1", student_1)
print_result("student 2", student_2)
print_result("student 3", student_3)
print_result("student 4", student_4)
print_result("student 5", student_5)

Advice for student 1:
Decision:
You are at high risk. Prioritise urgent revision and seek academic support

Explanation:
Risk level: high_risk
Reasoning: Very low attendance, Low coursework score, Deadline very close, Subject perceived as difficult
Confidence: 1.0

Probabilities:
{'high_risk': 0.55, 'medium_risk': 0.28, 'low_risk': 0.17}
----------------------------------------------------------
Advice for student 2:
Decision:
You are at high risk. Prioritise urgent revision and seek academic support

Explanation:
Risk level: high_risk
Reasoning: Very low attendance, Low coursework score, Deadline very close
Confidence: 0.9

Probabilities:
{'high_risk': 0.55, 'medium_risk': 0.28, 'low_risk': 0.17}
----------------------------------------------------------
Advice for student 3:
Decision:
You are at moderate risk. Improve consistency and focus on weak areas

Explanation:
Risk level: medium_risk
Reasoning: Moderate attendance, Average coursework, Deadline approaching
Confidence: 0.9

Prob